# Gap Finder agent, step by step

This notebook runs one real Gap Finder session one step at a time, in the order of the
[Gap Finder Agent Flow](https://claude.ai/artifact/DE2JwSjN6JESjSZbeCpKyq) canvas. Each cell calls the real function
from `gapfinder/agent/` and shows what it read, what it returned and how long it took.

- **Part 1** calls each step directly, so you can see every input and output.
- **Part 2** runs the same session through the real LangGraph agent, with its two pauses.

The notebook only reads from Neo4j and Pinecone. Part 2 keeps its session in memory, so nothing is written to
`data/sessions.sqlite`. A full run makes about 8 small OpenAI requests.

**Legend** (same as the canvas): `UI` screen · `AI` OpenAI · `G` Neo4j · `V` Pinecone · `fx` plain Python · `II` pause for the parent · `?` decision

## 0. Setup

In [ ]:
import json
import sys
import time
import uuid
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from gapfinder.agent import nodes as n, scoring
from gapfinder.agent.resources import fast_model, neo4j, write_model
from gapfinder.config import CONFIDENCE_THRESHOLD, MAX_ROUNDS, MODEL_FAST, MODEL_WRITE
from gapfinder.graph import queries as Q
from gapfinder.vectors import retrieve


def short(value, limit=160):
    """Shorten long strings so outputs stay readable."""
    if isinstance(value, str):
        return value if len(value) <= limit else value[:limit] + "…"
    if isinstance(value, list):
        return [short(v, limit) for v in value]
    if isinstance(value, dict):
        return {k: short(v, limit) for k, v in value.items()}
    return value


def show(label, value):
    print(f"{label}:")
    print(json.dumps(short(value), indent=2, ensure_ascii=False, default=str))


def step(tag, name, fn, state):
    """Run one agent step, print the fields it returned, and merge them into the state."""
    t = time.perf_counter()
    out = fn(state)
    ms = (time.perf_counter() - t) * 1000
    print(f"[{tag}] {name}  ({ms:,.0f} ms)  returned: {', '.join(out) or 'nothing'}\n")
    state.update(out)
    for key, value in out.items():
        show(f"  {key}", value)
    return out


print("models:", MODEL_FAST, "/", MODEL_WRITE)
print("neo4j:", neo4j().execute_query("MATCH (s:Standard {is_ccss: true}) RETURN count(s) AS c").records[0]["c"], "Common Core standards")
index, _ = retrieve._clients()
print("pinecone:", {ns: s.vector_count for ns, s in index.describe_index_stats().namespaces.items()})

### The example and the simulated child

Change these to try another problem. The simulated child answers correctly for the skills in `KNOWS` (using the
verified answer keys) and says "didn't know" for everything else, as in `scripts/smoke_app.py`.

In [ ]:
PROBLEM = "3/4 ÷ 1/8 = ?"
GRADE = "6"
STATE = "Texas"                       # None means Common Core
KNOWS = ["4.NF.B.4", "3.NF.A.1"]       # skills the simulated child has mastered


def child_answer(check):
    """The simulated child's answer to one check problem. None means "didn't know"."""
    knows_it = any(check["skill_code"] == k or check["skill_code"].startswith(k + ".") for k in KNOWS)
    return check["answer_key"] if knows_it else None


def parent_pick(choices):
    """The simulated parent's choice on the confirm screen: the first option."""
    return choices[0]["code"]

---
# Part 1: each step by hand

## Request 1: "Find the gap"
The parent submits the state, grade and problem (wizard step 1).

In [ ]:
# [UI] Parent submits: the screen starts a session with these three fields.
state = {"state_code": STATE, "grade": GRADE, "problem_text": PROBLEM}
show("state sent by the screen", state)

### `intake` · fx
Trims the problem to 1,000 characters and resets the per-session fields.

In [ ]:
step("fx", "intake", n.intake, state);

### `describe_skill` · AI
Restates the problem as the skill it practices. Searching with words finds the right standard where bare arithmetic
doesn't. A non-math problem ends the run here.

In [ ]:
step("AI", "describe_skill", n.describe_skill, state)
print("\nnext:", n.route_after_describe(state))

### `match_standard` · V + AI
First, what Pinecone returns: standards and skill components in the child's grade window, scored by each standard's
best match. Then the step itself, which also asks the model to pick one and say how sure it is.

In [ ]:
print("grade window:", retrieve.grade_window(GRADE))
t = time.perf_counter()
hits = retrieve.match_standards(state["skill_description"], GRADE, top_k=30, limit=8)
print(f"[V] Pinecone, standards namespace ({(time.perf_counter() - t) * 1000:,.0f} ms)")
for h in hits:
    print(f"  {h['score']:.3f}  {h['code']:<12} {h['kind']:<16} {short(h['evidence'], 90)}")

In [ ]:
step("V+AI", "match_standard", n.match_standard, state);

### Confident? · ?
The agent continues if the model's confidence is at least the threshold. Otherwise it pauses for the parent.

In [ ]:
route = n.route_after_match(state)
print(f"confidence {state['confidence']} vs threshold {CONFIDENCE_THRESHOLD} -> next: {route}")

### `confirm_skill` · II (only when unsure)
In the real agent this step pauses (a LangGraph interrupt) and the screen asks the parent to choose between the top two
skills. Calling it directly would fail outside the graph, so this cell shows what the screen receives and applies the
simulated parent's pick the same way the step does. Part 2 shows the real pause.

In [ ]:
if route == "confirm_skill":
    choices = [{"code": c["code"], "text": c["text"]} for c in state["candidates"][:2]]
    show("[II] the screen shows", choices)
    chosen = parent_pick(choices)
    pick = next(c for c in choices if c["code"] == chosen)
    state.update({"target_code": pick["code"], "target_text": pick["text"], "confidence": 1.0})
    print("\nparent picked:", chosen)
else:
    print("Confident: no pause. Target:", state["target_code"], "-", short(state["target_text"], 100))

### `state_wording` · G
The state's version of the standard, for display only. It never steers the diagnosis (decision #17).

In [ ]:
print(Q.STATE_WORDING)
step("G", "state_wording", n.state_wording, state);

### `prereq_walk` · G
Earlier skills from the Coherence Map, up to 2 steps back. Only skills with at least 2 verified problems are kept, and
at most 3. Sub-standards count toward their parent standard.

In [ ]:
print(Q.PREREQUISITES)
show("all candidates before filtering", Q.prerequisites(neo4j(), state["target_code"], limit=10))
print()
step("G", "prereq_walk", n.prereq_walk, state);

### `build_checks` · G
Two verified problems per skill, from the teacher-reviewed bank. Never generated. If no skill has verified problems,
the run ends with "can't check this skill yet".

In [ ]:
print(Q.CHECK_PROBLEMS)
step("G", "build_checks", n.build_checks, state)
route = n.route_after_checks(state)
print("\nnext:", route)
if route == "cannot_check":
    step("G", "cannot_check", n.cannot_check, state)
    print("\nThe run ends here. Change PROBLEM and run Part 1 again.")

### `await_answers` · II
The agent pauses while the parent runs the check, one problem per screen. The screen gets the problems without their
answer keys. As with `confirm_skill`, this cell shows what the screen receives and applies the simulated child's
answers. Part 2 shows the real pause.

In [ ]:
def run_check_round(state):
    this_round = [c for c in state["checks"] if c["round"] == state["round"]]
    show(f"[II] round {state['round']}: the screen shows", [{"id": c["id"], "text": c["text"], "kind": c["kind"]} for c in this_round])
    answers = {c["id"]: child_answer(c) for c in this_round}
    print()
    for c in this_round:
        print(f"  {c['skill_code']:<12} {c['id']:<22} child says {answers[c['id']]!r:<10} (key {c['answer_key']})")
    state["answers"] = {**state.get("answers", {}), **answers}


run_check_round(state)

## Request 3: "Submit answers"
### `score_answers` · fx
Compares each answer with its key using exact fractions (3/6, 1/2 and 0.5 are the same). Two right is **solid**, two
wrong is a **gap**, one of each is **unclear**. No AI.

In [ ]:
step("fx", "score_answers", n.score_answers, state);

### Clear result? · ? and `narrow` · fx
If a skill is unclear, or a skill below a gap still needs checking, and this was round 1, the agent runs a second round:
`narrow` picks the skills, then `build_checks`, the check and `score_answers` run again. At most 2 rounds.

In [ ]:
route = n.route_after_score(state)
print(f"round {state['round']} of at most {MAX_ROUNDS} -> next: {route}")
while route == "narrow":
    step("fx", "narrow", n.narrow, state)
    if n.route_after_narrow(state) != "build_checks":
        break
    print()
    step("G", "build_checks (round 2)", n.build_checks, state)
    print()
    run_check_round(state)
    print()
    step("fx", "score_answers (round 2)", n.score_answers, state)
    route = n.route_after_score(state)
    print("\nnext:", route)

### `decide_gap` · fx
Fixed rules: the likely gap is the most foundational skill marked gap whose own earlier skills are solid. High
confidence is a clear gap with solid earlier skills, medium is a gap found in round 2, low is still unclear.

In [ ]:
step("fx", "decide_gap", n.decide_gap, state);

### `retrieve_lessons` · G + V
Lessons for the gap skill come from the graph, then Pinecone ranks them against the skill. The top 3 are kept.

In [ ]:
print(Q.LESSONS)
found = Q.lessons(neo4j(), state["gap"]["code"], limit=8)
order = retrieve.rank_by_similarity([l["id"] for l in found], f"{state['skill_description']}. {state['gap_text']}")
titles = {l["id"]: l["title"] for l in found}
print(f"[G] {len(found)} lessons from the graph, [V] ranked by similarity:")
for i, score in order:
    print(f"  {score:.3f}  {i:<26} {titles[i]}")
print()
step("G+V", "retrieve_lessons", n.retrieve_lessons, state);

### `generate_result` · AI
The model writes the result from these facts only: the check results, the gap and the lessons. Below are the exact
facts it receives.

In [ ]:
print(n._facts(state))
print()
step("AI", "generate_result", n.generate_result, state);

### `grade` · fx + AI, then Grounded? · ?
First, plain Python checks that every lesson id is in the list and exists in Neo4j. Then one model call checks that
every claim is supported by the facts. A failure rewrites once; a second failure abstains.

In [ ]:
step("fx+AI", "grade", n.grade, state)
route = n.route_after_grade(state)
print("\nnext:", route)
if route == "generate_result":
    step("AI", "generate_result (rewrite)", n.generate_result, state)
    step("fx+AI", "grade (rewrite)", n.grade, state)
    route = n.route_after_grade(state)
    print("\nnext:", route)
if route == "abstain":
    step("fx", "abstain", n.abstain, state)

### Result screen · UI

In [ ]:
r = state["result"]
print("Likely gap:", state["gap"])
print("Headline:", r.get("headline"))
print("Why:", r.get("why"))
print("Lessons:")
for l in state["lessons"]:
    print(f"  grade {l['grade']} unit {l['unit']} lesson {l['lesson']}: {l['title']}  {l['url']}")

## Request 4: "Ask why"
The real chat runs as one function, `gapfinder.agent.followup.ask_why`, which reads the saved session. Part 1 has no
saved session, so these cells repeat its four parts with the same queries and prompts. Part 2 calls the real function.

In [ ]:
from gapfinder.agent.followup import NO_SOURCE, ChatAnswer

QUESTION = "How do I explain it?"
gap_code = state["gap"]["code"]

# [V+G] context: closest lesson and problem text, plus the skill's earlier and later skills
t = time.perf_counter()
hits = retrieve.search(QUESTION + " " + state.get("gap_text", ""), namespace="curriculum", top_k=4)
before = Q.prerequisites(neo4j(), gap_code, limit=3)
after = Q.next_skills(neo4j(), gap_code)
context = "\n".join([
    n._facts(state),
    *[f"Related {h['kind']} {h['id']}: {h.get('text', '')}" for h in hits],
    *[f"Comes before {gap_code}: {p['code']} {p['text']}" for p in before],
    *[f"{gap_code} leads to: {x['code']} {x['text']}" for x in after],
])
print(f"[V+G] context ({(time.perf_counter() - t) * 1000:,.0f} ms)\n")
print(context)

In [ ]:
# [AI] answer from the context only
t = time.perf_counter()
chat = write_model().with_structured_output(ChatAnswer).invoke(
    f"Context:\n{context}\n\nA parent asks: {QUESTION}\n\n"
    "Answer only from the context. Do not invent new practice problems or numbers. "
    "If the context doesn't answer the question, say so.")
print(f"[AI] answer ({(time.perf_counter() - t) * 1000:,.0f} ms)\n")
show("answer", chat.model_dump())

In [ ]:
# [AI] the same grounding check as the result
t = time.perf_counter()
verdict = fast_model().with_structured_output(n.Grade).invoke(
    f"Facts:\n{context}\n\nText to check:\n{chat.answer}\n\nIs every claim in the text supported by the facts?")
print(f"[AI] grounding check ({(time.perf_counter() - t) * 1000:,.0f} ms)\n")
show("verdict", verdict.model_dump())

In [ ]:
# [UI] reply on screen: ungrounded answers are replaced, and cited lessons must be in the context
allowed = {l["id"] for l in state["lessons"]} | {h["id"] for h in hits if h.get("kind") == "lesson"}
cited = [i for i in chat.lesson_ids if i in allowed]
print("Reply:", chat.answer if verdict.grounded else NO_SOURCE)
print("Cited lessons:", cited if verdict.grounded else [])

---
# Part 2: the same session through the real agent

This compiles the real graph from `gapfinder/agent/graph.py` with an in-memory checkpointer instead of
`data/sessions.sqlite`, then streams it. It stops at the two real pauses and resumes with the simulated parent's input,
just as the Streamlit screens do.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from gapfinder.agent import followup
from gapfinder.agent.graph import build

live = build(MemorySaver())
config = {"configurable": {"thread_id": f"notebook-{uuid.uuid4().hex[:8]}"}}


def run_until_pause(payload):
    """Stream the graph and print each step as it finishes. Returns the pause, or None when the run ends."""
    t0 = time.perf_counter()
    last = t0
    for update in live.stream(payload, config, stream_mode="updates"):
        for node, out in update.items():
            now = time.perf_counter()
            if node == "__interrupt__":
                print(f"  {'(pause)':<18} {(now - t0) * 1000:7,.0f} ms total")
                return out[0].value
            print(f"  {node:<18} {(now - last) * 1000:7,.0f} ms   {', '.join(out or {})}")
            last = now
    print(f"  {'(end)':<18} {(time.perf_counter() - t0) * 1000:7,.0f} ms total")
    return None

In [ ]:
print("Request 1: find the gap")
pause = run_until_pause({"state_code": STATE, "grade": GRADE, "problem_text": PROBLEM})

while pause:
    if pause["type"] == "confirm_skill":
        print("\nRequest 2: the parent picks a skill")
        pause = run_until_pause(Command(resume=parent_pick(pause["choices"])))
    else:
        print(f"\nRequest 3: answers for round {pause['round']} ({len(pause['problems'])} problems)")
        checks = {c["id"]: c for c in live.get_state(config).values["checks"]}
        answers = {p["id"]: child_answer(checks[p["id"]]) for p in pause["problems"]}
        pause = run_until_pause(Command(resume=answers))

In [ ]:
final = live.get_state(config).values
print("Likely gap:", final.get("gap"))
show("result", final.get("result"))

### "Ask why" through the real function
`ask_why` normally reads the session from `data/sessions.sqlite`. For this notebook only, it is pointed at the
in-memory agent above. No file is changed.

In [ ]:
followup.agent = lambda: live
t = time.perf_counter()
turn = followup.ask_why(config["configurable"]["thread_id"], QUESTION)
print(f"ask_why ({(time.perf_counter() - t) * 1000:,.0f} ms)\n")
show("turn", turn)

### Do both runs agree?

In [ ]:
rows = [
    ("target standard", state["target_code"], final.get("target_code")),
    ("gap", state["gap"]["code"], final.get("gap", {}).get("code")),
    ("gap confidence", state["gap"]["confidence"], final.get("gap", {}).get("confidence")),
    ("lessons", [l["id"] for l in state["lessons"]], [l["id"] for l in final.get("lessons", [])]),
]
for name, part1, part2 in rows:
    print(f"{'same' if part1 == part2 else 'DIFFERENT':<10} {name:<16} part 1: {part1}   part 2: {part2}")
print("\nModel steps can differ slightly between runs. The graph lookups, scoring and gap rules are deterministic.")